In [ ]:
from google.colab import files

files.download("generate_data.py")

FileNotFoundError: Cannot find file: generate_data.py

In [ ]:
import os

print("Current folder:", os.getcwd())
print("File exists:", os.path.exists("generate_data.py"))

Current folder: /content
File exists: False


In [ ]:
%%writefile generate_data.py

import numpy as np
import pandas as pd

# Duka Sales Generator
rng = np.random.default_rng(2026)
n = 1000

products = {
    "Phone": 15000,
    "Laptop": 65000,
    "Headphones": 2500,
    "Tablet": 28000,
    "Charger": 1200
}

sales = pd.DataFrame({
    "order_id": range(1001, 1001+n),
    "order_date": pd.to_datetime("2026-01-01") + pd.to_timedelta(rng.integers(0, 180, n), unit="D"),
    "branch": rng.choice(["Nairobi", "Mombasa", "Kisumu", "Nakuru", "Eldoret"], n, p=[0.4, 0.2, 0.15, 0.15, 0.1]),
    "product": rng.choice(list(products), n),
    "quantity": rng.integers(1, 6, n),
    "channel": rng.choice(["In-store", "Online", "Phone order"], n, p=[0.55, 0.35, 0.10]),
})

sales["unit_price"] = sales["product"].map(products)
sales.to_csv("duka_sales.csv", index=False)

# Agent Network Generator
rng = np.random.default_rng(44)
n = 600

agents = pd.DataFrame({
    "agent_id": [f"AG{i:03d}" for i in range(1, 21)],
    "county": rng.choice(["Nairobi", "Kiambu", "Nakuru", "Mombasa", "Kisumu"], 20),
    "onboarded": pd.date_range("2024-01-01", periods=20, freq="MS").strftime("%Y-%m-%d")
})

txns = pd.DataFrame({
    "txn_id": [f"TX{i:05d}" for i in range(1, n+1)],
    "agent_id": rng.choice(agents["agent_id"], n),
    "txn_type": rng.choice(["Deposit", "withdrawal", "DEPOSIT ", "Withdrawal", "deposit"], n),
    "amount": rng.lognormal(7.5, 0.8, n).round(0).astype(str),
    "txn_date": (pd.to_datetime("2026-03-01") + pd.to_timedelta(rng.integers(0, 60, n), unit="D")).strftime("%Y-%m-%d")
})

idx = rng.choice(n, 30, replace=False)
txns.loc[idx[:15], "amount"] = np.nan
txns.loc[idx[15:20], "amount"] = "N/A"
txns.loc[idx[20:25], "amount"] = "250000"
txns.loc[idx[25:], "txn_date"] = txns.loc[idx[25:], "txn_date"].str.replace("-", "/")

txns = pd.concat([txns, txns.sample(12, random_state=1)], ignore_index=True)
txns.loc[rng.choice(len(txns), 8, replace=False), "agent_id"] = "AG099"

agents.to_csv("agents.csv", index=False)
txns.to_csv("agent_transactions.csv", index=False)

print("All three datasets generated successfully!")

Writing generate_data.py


In [ ]:
%run generate_data.py

All three datasets generated successfully!


In [ ]:
import pandas as pd
import numpy as np

agents = pd.read_csv("agents.csv")
txns = pd.read_csv("agent_transactions.csv")

print("Agents dataset:")
display(agents.head())

print("Transactions dataset:")
display(txns.head())

Agents dataset:


,agent_id,county,onboarded
0,AG001,Mombasa,2024-01-01
1,AG002,Nairobi,2024-02-01
2,AG003,Kisumu,2024-03-01
3,AG004,Kiambu,2024-04-01
4,AG005,Mombasa,2024-05-01


Transactions dataset:


,txn_id,agent_id,txn_type,amount,txn_date
0,TX00001,AG009,deposit,353.0,2026-03-03
1,TX00002,AG020,deposit,1238.0,2026-04-27
2,TX00003,AG008,DEPOSIT,9129.0,2026-04-29
3,TX00004,AG009,deposit,652.0,2026-04-24
4,TX00005,AG009,withdrawal,478.0,2026-04-17


In [ ]:
print("AGENTS DATASET")
print("Shape:", agents.shape)
agents.info()

print("\nTRANSACTIONS DATASET")
print("Shape:", txns.shape)
txns.info()

AGENTS DATASET
Shape: (20, 3)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   agent_id   20 non-null     object
 1   county     20 non-null     object
 2   onboarded  20 non-null     object
dtypes: object(3)
memory usage: 612.0+ bytes

TRANSACTIONS DATASET
Shape: (612, 5)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 612 entries, 0 to 611
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   txn_id    612 non-null    object 
 1   agent_id  612 non-null    object 
 2   txn_type  612 non-null    object 
 3   amount    590 non-null    float64
 4   txn_date  612 non-null    object 
dtypes: float64(1), object(4)
memory usage: 24.0+ KB


In [ ]:
print("Missing values in agents:")
display(agents.isnull().sum())

print("Missing values in transactions:")
display(txns.isnull().sum())

Missing values in agents:


,0
agent_id,0
county,0
onboarded,0


Missing values in transactions:


,0
txn_id,0
agent_id,0
txn_type,0
amount,22
txn_date,0


In [ ]:
print("Total transaction rows:", len(txns))

print("Duplicate transaction IDs:",
      txns["txn_id"].duplicated().sum())

Total transaction rows: 612
Duplicate transaction IDs: 12


In [ ]:
before = len(txns)

txns = txns.drop_duplicates(subset="txn_id", keep="first")

after = len(txns)

print("Rows before cleaning:", before)
print("Rows after cleaning:", after)
print("Duplicates removed:", before - after)

Rows before cleaning: 612
Rows after cleaning: 600
Duplicates removed: 12


In [ ]:
print(txns["txn_type"].value_counts(dropna=False))

txn_type
deposit       138
withdrawal    128
Deposit       125
DEPOSIT       110
Withdrawal     99
Name: count, dtype: int64


In [ ]:
txns["txn_type"] = (
    txns["txn_type"]
    .str.strip()
    .str.lower()
    .str.capitalize()
)

print(txns["txn_type"].value_counts())

txn_type
Deposit       373
Withdrawal    227
Name: count, dtype: int64


In [ ]:
print("Amount data type:", txns["amount"].dtype)

print("\nAmount summary:")
display(txns["amount"].describe())

print("\nMissing amounts:", txns["amount"].isna().sum())

Amount data type: float64

Amount summary:


,amount
count,580.000000
mean,4573.960345
std,23030.332197
min,109.000000
25%,1077.000000
50%,1776.500000
75%,3080.000000
max,250000.000000



Missing amounts: 20


In [ ]:
txns["amount"] = pd.to_numeric(txns["amount"], errors="coerce")

print("Amount data type:", txns["amount"].dtype)
print("Missing amounts:", txns["amount"].isna().sum())

Amount data type: float64
Missing amounts: 20


In [ ]:
# Identify records that have missing amounts
txns["amount_imputed"] = txns["amount"].isna()

# Calculate the median amount
median_amount = txns["amount"].median()

# Replace missing amounts with the median
txns["amount"] = txns["amount"].fillna(median_amount)

print("Median used:", median_amount)
print("Missing amounts remaining:", txns["amount"].isna().sum())
print("Imputed records:", txns["amount_imputed"].sum())

Median used: 1776.5
Missing amounts remaining: 0
Imputed records: 20


In [ ]:
Q1 = txns["amount"].quantile(0.25)
Q3 = txns["amount"].quantile(0.75)

IQR = Q3 - Q1

lower_limit = Q1 - 1.5 * IQR
upper_limit = Q3 + 1.5 * IQR

print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)
print("Lower limit:", lower_limit)
print("Upper limit:", upper_limit)

Q1: 1105.0
Q3: 2868.5
IQR: 1763.5
Lower limit: -1540.25
Upper limit: 5513.75


In [ ]:
txns["is_outlier"] = (
    (txns["amount"] < lower_limit) |
    (txns["amount"] > upper_limit)
)

print("Total outliers:", txns["is_outlier"].sum())

print("\nOutlier transactions:")
display(
    txns.loc[
        txns["is_outlier"],
        ["txn_id", "agent_id", "txn_type", "amount"]
    ].head(10)
)

Total outliers: 53

Outlier transactions:


,txn_id,agent_id,txn_type,amount
2,TX00003,AG008,Deposit,9129.0
45,TX00046,AG016,Deposit,5632.0
55,TX00056,AG099,Deposit,8507.0
56,TX00057,AG017,Deposit,6006.0
57,TX00058,AG011,Withdrawal,8035.0
63,TX00064,AG007,Withdrawal,6566.0
65,TX00066,AG016,Withdrawal,7116.0
67,TX00068,AG019,Withdrawal,7962.0
69,TX00070,AG005,Deposit,5856.0
87,TX00088,AG014,Deposit,7072.0


In [ ]:
large_transactions = txns[txns["amount"] >= 250000]

display(
    large_transactions[
        ["txn_id", "agent_id", "txn_type", "amount", "is_outlier"]
    ]
)

print("Number of transactions:", len(large_transactions))

,txn_id,agent_id,txn_type,amount,is_outlier
306,TX00307,AG012,Deposit,250000.0,True
315,TX00316,AG014,Deposit,250000.0,True
424,TX00425,AG005,Withdrawal,250000.0,True
463,TX00464,AG009,Withdrawal,250000.0,True
594,TX00595,AG016,Deposit,250000.0,True


Number of transactions: 5


### Outlier Investigation
The IQR method identified 53 potential outliers. Five transactions had unusually high amounts of KES 250,000. These values were retained because an unusually high transaction is not necessarily an error. I created an `is_outlier` flag to identify unusual records and recommend manual verification before making any changes.

In [ ]:
print("Unique agent IDs in register:", agents["agent_id"].nunique())

print("Unique agent IDs in transactions:", txns["agent_id"].nunique())

print("\nAgents in register:")
print(sorted(agents["agent_id"].unique()))

print("\nAgent IDs in transactions:")
print(sorted(txns["agent_id"].unique()))

Unique agent IDs in register: 20
Unique agent IDs in transactions: 21

Agents in register:
['AG001', 'AG002', 'AG003', 'AG004', 'AG005', 'AG006', 'AG007', 'AG008', 'AG009', 'AG010', 'AG011', 'AG012', 'AG013', 'AG014', 'AG015', 'AG016', 'AG017', 'AG018', 'AG019', 'AG020']

Agent IDs in transactions:
['AG001', 'AG002', 'AG003', 'AG004', 'AG005', 'AG006', 'AG007', 'AG008', 'AG009', 'AG010', 'AG011', 'AG012', 'AG013', 'AG014', 'AG015', 'AG016', 'AG017', 'AG018', 'AG019', 'AG020', 'AG099']


In [ ]:
unknown_agents = txns[
    ~txns["agent_id"].isin(agents["agent_id"])
]

print("Unknown agent transactions:", len(unknown_agents))

display(unknown_agents.head())

Unknown agent transactions: 7


,txn_id,agent_id,txn_type,amount,txn_date,amount_imputed,is_outlier
55,TX00056,AG099,Deposit,8507.0,2026-04-01,False,True
68,TX00069,AG099,Withdrawal,1877.0,2026-04-24,False,False
77,TX00078,AG099,Deposit,1770.0,2026-04-06,False,False
237,TX00238,AG099,Deposit,4944.0,2026-04-06,False,False
267,TX00268,AG099,Deposit,3110.0,2026-04-09,False,False


In [ ]:
merged = txns.merge(
    agents[["agent_id", "county"]],
    on="agent_id",
    how="left",
    indicator=True
)

print("Rows before merge:", len(txns))
print("Rows after merge:", len(merged))

print("\nMerge results:")
print(merged["_merge"].value_counts())

Rows before merge: 600
Rows after merge: 600

Merge results:
_merge
both          593
left_only       7
right_only      0
Name: count, dtype: int64


In [ ]:
merged["county"] = merged["county"].fillna("Unknown")

print("Missing counties:", merged["county"].isna().sum())

print("\nCounty counts:")
print(merged["county"].value_counts())

Missing counties: 0

County counts:
county
Nairobi    180
Mombasa    133
Kisumu     111
Nakuru      99
Kiambu      70
Unknown      7
Name: count, dtype: int64


In [23]:
county_summary = merged.groupby("county").agg(
    total_value=("amount", "sum"),
    transaction_count=("txn_id", "count"),
    average_amount=("amount", "mean")
).reset_index()

display(county_summary)


,county,total_value,transaction_count,average_amount
0,Kiambu,411787.5,70,5882.678571
1,Kisumu,512795.0,111,4619.774775
2,Mombasa,517578.0,133,3891.563910
3,Nairobi,705626.0,180,3920.144444
4,Nakuru,513484.5,99,5186.712121
5,Unknown,27156.0,7,3879.428571


In [24]:
merged["txn_date"] = pd.to_datetime(
    merged["txn_date"],
    format="mixed",
    errors="coerce"
)

print("Date data type:", merged["txn_date"].dtype)
print("Missing dates:", merged["txn_date"].isna().sum())

Date data type: datetime64[ns]
Missing dates: 0


In [25]:
merged["week"] = merged["txn_date"].dt.to_period("W").astype(str)

weekly_summary = pd.pivot_table(
    merged,
    values="amount",
    index="week",
    columns="county",
    aggfunc="sum",
    fill_value=0
)

display(weekly_summary)

county,Kiambu,Kisumu,Mombasa,Nairobi,Nakuru,Unknown
week,,,,,,
2026-02-23/2026-03-01,0.0,1893.0,6759.0,7046.0,2743.5,0.0
2026-03-02/2026-03-08,18317.0,287933.0,285669.5,68967.0,56574.0,0.0
2026-03-09/2026-03-15,267771.5,23186.0,30477.5,52860.5,28730.0,0.0
2026-03-16/2026-03-22,22054.0,28125.0,25144.5,43107.5,39795.0,1639.0
2026-03-23/2026-03-29,11504.0,27077.0,17050.0,41373.5,18998.5,0.0
2026-03-30/2026-04-05,22609.0,41077.0,38344.0,282931.0,18145.0,8507.0
2026-04-06/2026-04-12,15030.0,15341.0,47092.5,56786.5,276609.0,9824.0
2026-04-13/2026-04-19,12265.0,29416.0,26730.0,56432.5,26152.5,0.0
2026-04-20/2026-04-26,24507.0,37439.0,16687.0,65187.0,30862.0,1877.0


In [26]:
%%writefile cleaning_pipeline.py

import pandas as pd

def clean_agent_data(txns, agents):
    """
    Clean agent transaction data and return:
    1. A cleaned transaction DataFrame
    2. A data-quality audit log
    """

    clean_df = txns.copy()
    audit_log = []

    # 1. Remove duplicate transaction IDs
    before = len(clean_df)
    clean_df = clean_df.drop_duplicates(
        subset="txn_id",
        keep="first"
    )
    removed = before - len(clean_df)

    audit_log.append({
        "issue": "Duplicate transactions",
        "dimension": "Uniqueness",
        "column": "txn_id",
        "rows affected": removed
    })

    # 2. Standardize transaction types
    clean_df["txn_type"] = (
        clean_df["txn_type"]
        .str.strip()
        .str.lower()
        .str.capitalize()
    )

    # 3. Convert amount to numeric
    clean_df["amount"] = pd.to_numeric(
        clean_df["amount"],
        errors="coerce"
    )

    # Flag missing amounts before filling them
    missing_mask = clean_df["amount"].isna()
    missing_count = missing_mask.sum()

    median_amount = clean_df["amount"].median()

    clean_df["amount_imputed"] = missing_mask
    clean_df["amount"] = clean_df["amount"].fillna(median_amount)

    audit_log.append({
        "issue": "Missing or invalid amounts",
        "dimension": "Completeness",
        "column": "amount",
        "rows affected": int(missing_count)
    })

    # 4. Convert transaction dates
    clean_df["txn_date"] = pd.to_datetime(
        clean_df["txn_date"],
        format="mixed",
        errors="coerce"
    )

    invalid_dates = clean_df["txn_date"].isna().sum()

    audit_log.append({
        "issue": "Invalid transaction dates",
        "dimension": "Validity",
        "column": "txn_date",
        "rows affected": int(invalid_dates)
    })

    # 5. Detect outliers using IQR
    q1 = clean_df["amount"].quantile(0.25)
    q3 = clean_df["amount"].quantile(0.75)
    iqr = q3 - q1

    lower_limit = q1 - 1.5 * iqr
    upper_limit = q3 + 1.5 * iqr

    clean_df["is_outlier"] = (
        (clean_df["amount"] < lower_limit) |
        (clean_df["amount"] > upper_limit)
    )

    outlier_count = clean_df["is_outlier"].sum()

    audit_log.append({
        "issue": "Potential outliers",
        "dimension": "Accuracy",
        "column": "amount",
        "rows affected": int(outlier_count)
    })

    # 6. Merge transactions with agent register
    clean_df = clean_df.merge(
        agents[["agent_id", "county"]],
        on="agent_id",
        how="left"
    )

    unknown_count = clean_df["county"].isna().sum()
    clean_df["county"] = clean_df["county"].fillna("Unknown")

    audit_log.append({
        "issue": "Unknown agent IDs",
        "dimension": "Consistency",
        "column": "agent_id",
        "rows affected": int(unknown_count)
    })

    return clean_df, audit_log

Writing cleaning_pipeline.py


In [27]:
from cleaning_pipeline import clean_agent_data

# Load the original, uncleaned transaction data again
raw_txns = pd.read_csv("agent_transactions.csv")

# Run the reusable cleaning function
clean_df, log = clean_agent_data(raw_txns, agents)

print("Original transaction rows:", len(raw_txns))
print("Cleaned transaction rows:", len(clean_df))
print("Missing amounts:", clean_df["amount"].isna().sum())

Original transaction rows: 612
Cleaned transaction rows: 600
Missing amounts: 0


In [28]:
audit_table = pd.DataFrame(log)

display(audit_table)

,issue,dimension,column,rows affected
0,Duplicate transactions,Uniqueness,txn_id,12
1,Missing or invalid amounts,Completeness,amount,20
2,Invalid transaction dates,Validity,txn_date,0
3,Potential outliers,Accuracy,amount,53
4,Unknown agent IDs,Consistency,agent_id,7


In [30]:
import importlib
import cleaning_pipeline

importlib.reload(cleaning_pipeline)

from cleaning_pipeline import clean_agent_data

In [31]:
clean_df, log = clean_agent_data(raw_txns, agents)

audit_table = pd.DataFrame(log)

display(audit_table)

,issue,dimension,column,rows affected
0,Duplicate transactions,Uniqueness,txn_id,12
1,Transaction type standardization,Consistency,txn_type,376
2,Missing or invalid amounts,Completeness,amount,20
3,Invalid transaction dates,Validity,txn_date,0
4,Potential outliers,Accuracy,amount,53
5,Unknown agent IDs,Consistency,agent_id,7


In [32]:
clean_df["txn_type"].value_counts()

,count
txn_type,
Deposit,373
Withdrawal,227


In [33]:
clean_df.isnull().sum()

,0
txn_id,0
agent_id,0
txn_type,0
amount,0
txn_date,0
amount_imputed,0
is_outlier,0
county,0


In [34]:
clean_df["county"].value_counts()

,count
county,
Nairobi,180
Mombasa,133
Kisumu,111
Nakuru,99
Kiambu,70
Unknown,7


In [35]:
import os

# Create the outputs folder if it doesn't exist
os.makedirs("outputs", exist_ok=True)

# Make a copy for export
export_df = clean_df.copy()

# Format dates as YYYY-MM-DD
export_df["txn_date"] = export_df["txn_date"].dt.strftime("%Y-%m-%d")

# Save the cleaned data
export_df.to_csv(
    "outputs/agent_transactions_clean.csv",
    index=False
)

print("Cleaned transaction file saved successfully!")

Cleaned transaction file saved successfully!


In [36]:
check_df = pd.read_csv("outputs/agent_transactions_clean.csv")

print("Rows:", check_df.shape[0])
print("Columns:", check_df.shape[1])

display(check_df.head())

Rows: 600
Columns: 8


,txn_id,agent_id,txn_type,amount,txn_date,amount_imputed,is_outlier,county
0,TX00001,AG009,Deposit,353.0,2026-03-03,False,False,Nakuru
1,TX00002,AG020,Deposit,1238.0,2026-04-27,False,False,Mombasa
2,TX00003,AG008,Deposit,9129.0,2026-04-29,False,True,Kisumu
3,TX00004,AG009,Deposit,652.0,2026-04-24,False,False,Nakuru
4,TX00005,AG009,Withdrawal,478.0,2026-04-17,False,False,Nakuru


In [37]:
county_summary = clean_df.groupby("county").agg(
    total_value=("amount", "sum"),
    transaction_count=("txn_id", "count"),
    average_amount=("amount", "mean")
).reset_index()

county_summary = county_summary.sort_values(
    by="total_value",
    ascending=False
)

display(county_summary)

,county,total_value,transaction_count,average_amount
3,Nairobi,705626.0,180,3920.144444
2,Mombasa,517578.0,133,3891.563910
4,Nakuru,513484.5,99,5186.712121
1,Kisumu,512795.0,111,4619.774775
0,Kiambu,411787.5,70,5882.678571
5,Unknown,27156.0,7,3879.428571


### County Performance Analysis

Nairobi recorded the highest total transaction value at KES 705,626 across 180 transactions. Kiambu had the highest average transaction amount at KES 5,882.68, despite recording only 70 transactions. Mombasa generated KES 517,578 from 133 transactions. The Unknown category contained 7 transactions worth KES 27,156, showing the importance of investigating agent IDs that are missing from the agent register.

**Recommendation:** Investigate the seven unknown agent records and maintain accurate agent registration information to improve data quality and reporting.

In [38]:
clean_df["week"] = (
    clean_df["txn_date"]
    .dt.to_period("W")
    .astype(str)
)

clean_df[["txn_date", "week"]].head()

,txn_date,week
0,2026-03-03,2026-03-02/2026-03-08
1,2026-04-27,2026-04-27/2026-05-03
2,2026-04-29,2026-04-27/2026-05-03
3,2026-04-24,2026-04-20/2026-04-26
4,2026-04-17,2026-04-13/2026-04-19


In [39]:
weekly_pivot = pd.pivot_table(
    clean_df,
    values="amount",
    index="week",
    columns="county",
    aggfunc="sum",
    fill_value=0
)

display(weekly_pivot)

county,Kiambu,Kisumu,Mombasa,Nairobi,Nakuru,Unknown
week,,,,,,
2026-02-23/2026-03-01,0.0,1893.0,6759.0,7046.0,2743.5,0.0
2026-03-02/2026-03-08,18317.0,287933.0,285669.5,68967.0,56574.0,0.0
2026-03-09/2026-03-15,267771.5,23186.0,30477.5,52860.5,28730.0,0.0
2026-03-16/2026-03-22,22054.0,28125.0,25144.5,43107.5,39795.0,1639.0
2026-03-23/2026-03-29,11504.0,27077.0,17050.0,41373.5,18998.5,0.0
2026-03-30/2026-04-05,22609.0,41077.0,38344.0,282931.0,18145.0,8507.0
2026-04-06/2026-04-12,15030.0,15341.0,47092.5,56786.5,276609.0,9824.0
2026-04-13/2026-04-19,12265.0,29416.0,26730.0,56432.5,26152.5,0.0
2026-04-20/2026-04-26,24507.0,37439.0,16687.0,65187.0,30862.0,1877.0


### Weekly County Transaction Analysis

The weekly pivot table compares transaction values across counties over time. It helps identify changes in transaction activity and unusually high weekly totals. Nairobi recorded KES 282,931 during the week of 30 March to 5 April 2026, while Nakuru recorded KES 276,609 during the week of 6 April to 12 April 2026. These unusually high totals should be interpreted carefully because the dataset contains transactions flagged as potential outliers. The outlier records were retained for investigation rather than automatically removed.

# DATA QUALITY REPORT
## Agent Network Data Cleaning

### 1. Completeness
The original transaction data contained 20 missing or invalid amount values after duplicate removal. These values were replaced with the median transaction amount of KES 1,776.50. An `amount_imputed` flag was added to identify records where amounts were filled in.

### 2. Accuracy
The IQR method identified 53 potential outliers. Five transactions had unusually high amounts of KES 250,000. These transactions were retained and flagged using the `is_outlier` column because unusually high values are not automatically incorrect.

### 3. Consistency
Transaction types were standardized into two categories: `Deposit` and `Withdrawal`. The agent register was also merged with transaction data using `agent_id`. Seven transactions belonged to unknown agent ID `AG099` and were assigned the county label `Unknown`.

### 4. Validity
Transaction amounts were converted into numeric values, and transaction dates were converted into valid datetime values. No invalid dates remained after cleaning.

### 5. Uniqueness
The original dataset contained 612 transaction records. Twelve duplicate transaction IDs were removed, leaving 600 unique transactions.

### 6. Timeliness
Transaction dates covered the period from 1 March to 29 April 2026. Dates were converted into a consistent format and used to produce weekly county summaries.

### Final Assessment
The cleaning process improved the reliability and consistency of the agent transaction dataset. The final dataset contains 600 records and eight columns, with no missing values. Further investigation is recommended for the seven unknown agent records and the 53 potential outliers.

### Data Privacy and the Kenya Data Protection Act, 2019

The Kenya Data Protection Act, 2019 protects personal information belonging to individuals. In an agent network, information such as phone numbers, names and identification details should be handled responsibly. Organisations should collect only information necessary for a legitimate purpose, explain how it will be used, keep it accurate, restrict access to authorised people and protect it against unauthorised disclosure. Personal information should not be kept longer than necessary. Individuals also have rights to access their personal data and request correction of inaccurate information. In this project, agent IDs and county information were used for analysis, and unnecessary personal details should be excluded from shared reports and exported datasets.

In [40]:
%%writefile requirements.txt

pandas
numpy
openpyxl

Writing requirements.txt


In [41]:
import os

for root, dirs, files in os.walk("."):
    dirs[:] = [d for d in dirs if d not in [".config", "__pycache__"]]
    for file in files:
        print(os.path.join(root, file))

./agent_transactions.csv
./duka_sales.csv
./requirements.txt
./agents.csv
./cleaning_pipeline.py
./generate_data.py
./outputs/agent_transactions_clean.csv
./sample_data/anscombe.json
./sample_data/README.md
./sample_data/california_housing_train.csv
./sample_data/mnist_test.csv
./sample_data/mnist_train_small.csv
./sample_data/california_housing_test.csv


In [42]:
import pandas as pd
import numpy as np
import os

sales = pd.read_csv(
    "duka_sales.csv",
    parse_dates=["order_date"]
)

sales["revenue"] = sales["quantity"] * sales["unit_price"]

sales["price_band"] = np.where(
    sales["unit_price"] >= 28000,
    "Premium",
    "Value"
)

premium_online = sales[
    (sales["price_band"] == "Premium") &
    (sales["channel"] == "Online") &
    (sales["branch"].isin(["Nairobi", "Mombasa"]))
]

os.makedirs("outputs", exist_ok=True)

premium_online.to_csv(
    "outputs/premium_online_coast_capital.csv",
    index=False
)

print("Exported records:", len(premium_online))

Exported records: 83


In [43]:
import pandas as pd
import os

# Calculate branch performance
branch_summary = sales.groupby("branch").agg(
    total_revenue=("revenue", "sum"),
    total_orders=("order_id", "count")
).reset_index()

# Create branch-to-region lookup
region_lookup = pd.DataFrame({
    "branch": ["Nairobi", "Mombasa", "Kisumu", "Nakuru", "Eldoret"],
    "region": ["Central", "Coast", "Western", "Rift Valley", "Rift Valley"]
})

# Create Excel folder
os.makedirs("excel", exist_ok=True)

# Save workbook
file_path = "excel/duka_branch_summary.xlsx"

with pd.ExcelWriter(file_path, engine="openpyxl") as writer:
    sales.to_excel(writer, sheet_name="SalesData", index=False)
    branch_summary.to_excel(writer, sheet_name="BranchSummary", index=False)
    region_lookup.to_excel(writer, sheet_name="RegionLookup", index=False)

print("Excel workbook created successfully!")

Excel workbook created successfully!


In [44]:
from openpyxl import load_workbook

file_path = "excel/duka_branch_summary.xlsx"

wb = load_workbook(file_path)

ws = wb["BranchSummary"]

# Add formula headings
ws["D1"] = "SUMIFS Revenue"
ws["E1"] = "Region (XLOOKUP)"

# Add formulas for the five branches
for row in range(2, 7):
    ws[f"D{row}"] = (
        f'=SUMIFS(SalesData!$H:$H,SalesData!$C:$C,A{row})'
    )

    ws[f"E{row}"] = (
        f'=XLOOKUP(A{row},RegionLookup!$A:$A,RegionLookup!$B:$B,"Not Found")'
    )

wb.save(file_path)

print("SUMIFS and XLOOKUP formulas added successfully!")

SUMIFS and XLOOKUP formulas added successfully!


In [45]:
pivot_summary = pd.pivot_table(
    sales,
    values="revenue",
    index="branch",
    columns="channel",
    aggfunc="sum",
    fill_value=0,
    margins=True,
    margins_name="Grand Total"
)

display(pivot_summary)

channel,In-store,Online,Phone order,Grand Total
branch,,,,
Eldoret,4885300,3450700,508200,8844200
Kisumu,6468600,3370800,1028700,10868100
Mombasa,8039600,4086800,1877300,14003700
Nairobi,14808400,7527400,3770300,26106100
Nakuru,6767300,4350400,852700,11970400
Grand Total,40969200,22786100,8037200,71792500


In [46]:
with pd.ExcelWriter(
    "excel/duka_branch_summary.xlsx",
    engine="openpyxl",
    mode="a",
    if_sheet_exists="replace"
) as writer:

    pivot_summary.to_excel(
        writer,
        sheet_name="PivotTable"
    )

print("Pivot summary added to Excel workbook!")

Pivot summary added to Excel workbook!


In [48]:
from google.colab import files

files.download("excel/duka_branch_summary.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>